## import libraries

In [33]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnableLambda
import re

In [34]:
# 1. MODEL
# ============================================================

model = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [35]:
# ============================================================
# 2. DECOMPOSER
# ============================================================

decompose_prompt = PromptTemplate.from_template(
    """
Split the question into up to 3 concise sub-questions.

Return ONLY a numbered list:

1. ...
2. ...
3. ...

Question:
{question}
"""
)

decomposer = decompose_prompt | model

In [36]:
# ============================================================
# 3. PARSE NUMBERED SUB-QUESTIONS
# ============================================================

def parse_numbered_subquestions(message):

    text = getattr(message, "content", str(message)).strip()

    lines = text.splitlines()

    subquestions = []

    for line in lines:

        match = re.match(
            r"\s*\d+\s*[.)]\s*(.*\S.*)$",
            line
        )

        if match:
            subquestions.append(
                match.group(1).strip()
            )

    # Fallback
    if not subquestions and text:
        subquestions = [text]

    # Make sure we never exceed 3
    return subquestions[:3]


parse_subquestions = RunnableLambda(
    parse_numbered_subquestions
)

In [37]:
# ============================================================
# 4. ANSWER PROMPT
# ============================================================

answer_prompt = PromptTemplate.from_template(
    """
You are a concise technical assistant.

Answer the following sub-question.

Return exactly this format:

Answer: <one-line answer>
Steps:
- <step 1>
- <step 2>

Keep it short.

Sub-question:
{subq}
"""
)

answer_chain = answer_prompt | model

In [38]:
# 5. ANSWER EACH SUB-QUESTION
# ============================================================

def run_answers(subquestions):

    inputs = [
        {"subq": question}
        for question in subquestions
    ]

    # Run multiple questions efficiently
    outputs = answer_chain.batch(inputs)

    parsed_answers = []

    for output in outputs:

        text = getattr(
            output,
            "content",
            str(output)
        ).strip()

        answer = None
        steps = []

        for line in text.splitlines():

            if line.lower().startswith("answer:"):

                answer = line.split(
                    ":",
                    1
                )[1].strip()
            elif re.match(
                r"\s*[-•]\s+",
                line
            ):

                step = re.sub(
                    r"^\s*[-•]\s+",
                    "",
                    line
                ).strip()

                steps.append(step)

        parsed_answers.append(
            {
                "answer": answer or text,
                "steps": steps or ["No steps parsed"],
                "raw": text
            }
        )

    return parsed_answers


run_answers_runnable = RunnableLambda(
    run_answers
)

In [39]:
# 6. FORMAT SUB-ANSWERS
# ============================================================

def format_subanswers(answers):

    blocks = []

    for index, answer in enumerate(
        answers,
        start=1
    ):

        blocks.append(
            f"{index}. Answer: {answer['answer']}"
        )

        blocks.append("   Steps:")

        for step in answer["steps"]:

            blocks.append(
                f"   - {step}"
            )

    return "\n".join(blocks)


format_runnable = RunnableLambda(
    lambda answers: {
        "subanswers_text": format_subanswers(answers)
    }
)

In [40]:

# ============================================================
# 7. COMBINER
# ============================================================

combine_prompt = PromptTemplate.from_template(
    """
Synthesize a single concise final answer
from these sub-answer blocks.

Input:

{subanswers_text}

Return exactly three lines:

1) Final Answer: <one line>
2) Key points: - <p1>; - <p2>
3) Confidence: <low/medium/high>
"""
)


combiner = (
    format_runnable
    | combine_prompt
    | model
)

In [41]:
# ============================================================
# 8. COMPLETE LCEL PIPELINE
# ============================================================

pipeline = (
    decomposer
    | parse_subquestions
    | run_answers_runnable
    | combiner
)

In [42]:
question = (
        "How can I reduce latency in a web app "
        "that serves ML predictions?"
    )

final = pipeline.invoke(
        {"question": question}
    )

print("\nFINAL ANSWER\n")
print(final.content)


FINAL ANSWER

Final Answer: Optimizing model inference, network latency, database queries, and server-side processing can help reduce latency in complex models.

Key points:
- Optimize model inference using efficient algorithms and hardware acceleration.
- Minimize network latency by caching frequently accessed data and using CDNs.
- Reduce database query latency by indexing and optimizing database schema.

Confidence: High
